# V2 — Linear Probes and Cosine-Jaccard Cross-Validation

Trains Probe 1 (object vs checker) per concept per layer.
Computes cosine similarity matrix of probe weight vectors.
Cross-validates against Jaccard similarity from neuron-view masks.

**CPU only.** Reads V1 vectors H5 and neuron list CSVs.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["h5py", "numpy", "pandas", "scikit-learn", "matplotlib", "seaborn", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import ast as ast_mod
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from pathlib import Path

LANG = "P"
MODEL = "QW"
PREFIX = f"{LANG}_{MODEL}_"
MODEL_CONFIGS = {
    "QW": {"id": "Qwen/Qwen2.5-Coder-7B",                "n_layers": 28, "mlp_dim": 3584},
    "DS": {"id": "deepseek-ai/deepseek-coder-6.7b-base",  "n_layers": 32, "mlp_dim": 4096},
}
N_LAYERS = MODEL_CONFIGS[MODEL]["n_layers"]
MLP_DIM = MODEL_CONFIGS[MODEL]["mlp_dim"]
EPSILON = "0.5"
CONSISTENCY = "0.8"

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = Path("/content/drive/MyDrive/DATA/New-Atlas")
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/New-Atlas")

VECTORS_FILE = DATA_DIR / f"{PREFIX}V1_vectors.h5"

print(f"Cell: {LANG}_{MODEL} | Layers: {N_LAYERS}")
print(f"Vectors: {VECTORS_FILE}")
print(f"Exists: {VECTORS_FILE.exists()}")

In [ ]:
# Cell 2 – Load vectors and identify concepts

hf = h5py.File(str(VECTORS_FILE), "r")

obj_concepts = sorted(hf["object"].keys()) if "object" in hf else []
chk_concepts = sorted(hf["checker"].keys()) if "checker" in hf else []
testable = sorted(set(obj_concepts) & set(chk_concepts))

print(f"Object concepts: {len(obj_concepts)}")
print(f"Checker concepts: {len(chk_concepts)}")
print(f"Testable (both): {len(testable)}")

# Quick shape check
c = testable[0]
print(f"\nSample: {c}")
print(f"  Object L0: {hf[f'object/{c}/layer_0'].shape}")
print(f"  Checker L0: {hf[f'checker/{c}/layer_0'].shape}")

In [ ]:
# Cell 3 – Train probes: Probe 1 (object vs checker) per concept per layer

probe_results = []  # per concept per layer
weight_vectors = {}  # (concept, layer) -> unit weight vector

for concept in testable:
    for lid in range(N_LAYERS):
        obj_key = f"object/{concept}/layer_{lid}"
        chk_key = f"checker/{concept}/layer_{lid}"

        if obj_key not in hf or chk_key not in hf:
            continue

        X_obj = hf[obj_key][:].astype(np.float32)
        X_chk = hf[chk_key][:].astype(np.float32)
        n_obj, n_chk = len(X_obj), len(X_chk)

        if n_obj < 5 or n_chk < 5:
            continue

        X = np.vstack([X_obj, X_chk])
        y = np.array([1]*n_obj + [0]*n_chk)

        # Standardize
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Train probe
        clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")

        # Cross-val accuracy
        n_folds = min(5, n_obj, n_chk)
        if n_folds >= 2:
            scores = cross_val_score(clf, X_scaled, y, cv=n_folds, scoring="accuracy")
            acc = scores.mean()
            acc_std = scores.std()
        else:
            acc, acc_std = np.nan, np.nan

        # Train on all data for weight vector
        clf.fit(X_scaled, y)
        w = clf.coef_[0]  # (mlp_dim,)
        w_norm = w / (np.linalg.norm(w) + 1e-10)  # unit normalize

        weight_vectors[(concept, lid)] = w_norm

        # Top 10 neurons by |weight|
        top_idx = np.argsort(np.abs(w))[::-1][:10].tolist()

        probe_results.append({
            "concept": concept,
            "layer": lid,
            "accuracy": acc,
            "accuracy_std": acc_std,
            "n_object": n_obj,
            "n_checker": n_chk,
            "top10_neurons": str(top_idx),
        })

df_probes = pd.DataFrame(probe_results)
print(f"Probe results: {len(df_probes)} rows ({len(testable)} concepts x {N_LAYERS} layers)")
print(f"Mean accuracy: {df_probes['accuracy'].mean():.3f}")
print(f"Weight vectors stored: {len(weight_vectors)}")

In [ ]:
# Cell 4 – Probe accuracy by layer

fig, ax = plt.subplots(figsize=(10, 5))
layer_means = df_probes.groupby("layer")["accuracy"].mean()
layer_stds = df_probes.groupby("layer")["accuracy"].std()
ax.plot(layer_means.index, layer_means.values, "o-", linewidth=2)
ax.fill_between(layer_means.index,
                layer_means.values - layer_stds.values,
                layer_means.values + layer_stds.values, alpha=0.2)
ax.set_xlabel("Layer")
ax.set_ylabel("Probe 1 Accuracy (object vs checker)")
ax.set_title(f"{LANG}_{MODEL}: Probe accuracy by layer")
ax.axhline(0.5, color="gray", linestyle="--", label="chance")
ax.legend()
plt.tight_layout()
plt.savefig(DATA_DIR / f"{PREFIX}V2_probe_accuracy_by_layer.png", dpi=150)
plt.show()

peak_layer = layer_means.idxmax()
print(f"Peak probe accuracy: {layer_means.max():.3f} at layer {peak_layer}")

In [ ]:
# Cell 5 – Compute cosine similarity matrix D(L) at each layer

cosine_matrices = {}  # layer -> (n_concepts, n_concepts) matrix

for lid in range(N_LAYERS):
    concepts_at_layer = [c for c in testable if (c, lid) in weight_vectors]
    if len(concepts_at_layer) < 3:
        continue

    W = np.stack([weight_vectors[(c, lid)] for c in concepts_at_layer])  # (n, mlp_dim)
    # Cosine = dot product of unit vectors
    D = W @ W.T  # (n, n)
    cosine_matrices[lid] = (concepts_at_layer, D)

print(f"Cosine matrices computed for {len(cosine_matrices)} layers")

In [ ]:
# Cell 6 – Compute Jaccard similarity matrix J(L) from neuron lists

def load_neuron_list(data_dir, prefix, eps, cons, obj_type="both"):
    p1 = os.path.join(str(data_dir), f"{prefix}4_neuron_list_eps{eps}_cons{cons}_layers_part1_{obj_type}.csv")
    p2 = os.path.join(str(data_dir), f"{prefix}4_neuron_list_eps{eps}_cons{cons}_layers_part2_{obj_type}.csv")
    if os.path.exists(p1) and os.path.exists(p2):
        return pd.concat([pd.read_csv(p1), pd.read_csv(p2)], ignore_index=True)
    old = os.path.join(str(data_dir), f"{prefix}4_neuron_list_eps{eps}_cons{cons}_all_layers_{obj_type}.csv")
    if os.path.exists(old):
        return pd.read_csv(old)
    return None


df_nl = load_neuron_list(DATA_DIR, PREFIX, EPSILON, CONSISTENCY)
if df_nl is not None:
    print(f"Neuron list loaded: {len(df_nl)} rows")
else:
    print("WARNING: neuron list not found")


def get_concept_only_set(df_nl, obj_name, layer):
    """Get concept-only neuron set."""
    row = df_nl[(df_nl["object"] == obj_name) & (df_nl["layer"] == layer)]
    if len(row) == 0:
        return set()
    return set(ast_mod.literal_eval(row.iloc[0]["concept_only"]))


def jaccard(a, b):
    if len(a) == 0 and len(b) == 0:
        return 0.0
    return len(a & b) / len(a | b)


# Map concept names to neuron list object names
# Python: concept "Assert" -> neuron list "ast__Assert"
# Need to figure out the mapping
nl_objects = set(df_nl["object"].unique()) if df_nl is not None else set()

def concept_to_nl_key(concept):
    """Map V1 concept name to neuron list object key."""
    # Try direct prefixed matches
    for prefix in ["ast__", "builtin__", "rust__"]:
        if f"{prefix}{concept}" in nl_objects:
            return f"{prefix}{concept}"
    return None


# Compute Jaccard matrices
jaccard_matrices = {}  # layer -> (concepts, J matrix)

if df_nl is not None:
    for lid in cosine_matrices:
        concepts, D = cosine_matrices[lid]
        n = len(concepts)
        J = np.zeros((n, n))

        sets_cache = {}
        for i, c in enumerate(concepts):
            nl_key = concept_to_nl_key(c)
            if nl_key:
                sets_cache[i] = get_concept_only_set(df_nl, nl_key, lid)
            else:
                sets_cache[i] = set()

        for i in range(n):
            for j in range(i, n):
                j_val = jaccard(sets_cache[i], sets_cache[j])
                J[i, j] = j_val
                J[j, i] = j_val

        jaccard_matrices[lid] = J

    print(f"Jaccard matrices computed for {len(jaccard_matrices)} layers")

In [ ]:
# Cell 7 – Cross-validation: Cosine vs Jaccard correlation at each layer

correlations = []

for lid in sorted(cosine_matrices.keys()):
    if lid not in jaccard_matrices:
        continue

    concepts, D = cosine_matrices[lid]
    J = jaccard_matrices[lid]
    n = len(concepts)

    # Extract upper triangles (excluding diagonal)
    idx = np.triu_indices(n, k=1)
    d_upper = D[idx]
    j_upper = J[idx]

    # Filter out pairs where both Jaccard = 0 (no neuron data)
    mask = (j_upper > 0) | (d_upper != 0)
    if mask.sum() < 10:
        correlations.append({"layer": lid, "pearson_r": np.nan, "spearman_rho": np.nan, "n_pairs": mask.sum()})
        continue

    d_filt = d_upper[mask]
    j_filt = j_upper[mask]

    r_pearson, p_pearson = stats.pearsonr(j_filt, d_filt)
    r_spearman, p_spearman = stats.spearmanr(j_filt, d_filt)

    correlations.append({
        "layer": lid,
        "pearson_r": r_pearson,
        "spearman_rho": r_spearman,
        "p_pearson": p_pearson,
        "p_spearman": p_spearman,
        "n_pairs": int(mask.sum()),
    })

df_corr = pd.DataFrame(correlations)
print("Cosine-Jaccard correlation by layer:")
display(df_corr)

peak_r = df_corr["pearson_r"].max()
peak_layer = df_corr.loc[df_corr["pearson_r"].idxmax(), "layer"]
mean_r = df_corr["pearson_r"].mean()
print(f"\nPeak Pearson r: {peak_r:.3f} at layer {peak_layer}")
print(f"Mean Pearson r: {mean_r:.3f}")
print(f"Layers with r > 0.7: {(df_corr['pearson_r'] > 0.7).sum()}/{len(df_corr)}")

In [ ]:
# Cell 8 – Correlation by layer figure

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pearson r by layer
ax = axes[0]
ax.plot(df_corr["layer"], df_corr["pearson_r"], "o-", linewidth=2)
ax.axhline(0.7, color="red", linestyle="--", alpha=0.5, label="r = 0.7 target")
ax.axhline(0.0, color="gray", linestyle="-", alpha=0.3)
ax.set_xlabel("Layer")
ax.set_ylabel("Pearson r (Jaccard vs Cosine)")
ax.set_title(f"{LANG}_{MODEL}: Jaccard-Cosine Correlation")
ax.legend()

# Scatter at peak layer
ax = axes[1]
if int(peak_layer) in jaccard_matrices and int(peak_layer) in cosine_matrices:
    concepts, D = cosine_matrices[int(peak_layer)]
    J = jaccard_matrices[int(peak_layer)]
    n = len(concepts)
    idx = np.triu_indices(n, k=1)
    ax.scatter(J[idx], D[idx], alpha=0.3, s=10)
    ax.set_xlabel("Jaccard (neuron view)")
    ax.set_ylabel("Cosine (vector view)")
    ax.set_title(f"Layer {int(peak_layer)} scatter (r = {peak_r:.3f})")

plt.tight_layout()
plt.savefig(DATA_DIR / f"{PREFIX}V2_jaccard_cosine_cross_validation.png", dpi=150)
plt.show()

In [ ]:
# Cell 9 – Save results

# Probe results CSV
probe_path = DATA_DIR / f"{PREFIX}V2_probe_results.csv"
df_probes.to_csv(probe_path, index=False)
print(f"Saved: {probe_path}")

# Correlation CSV
corr_path = DATA_DIR / f"{PREFIX}V2_cosine_jaccard_correlation.csv"
df_corr.to_csv(corr_path, index=False)
print(f"Saved: {corr_path}")

# Weight vectors (numpy archive for reuse)
wv_path = DATA_DIR / f"{PREFIX}V2_weight_vectors.npz"
wv_dict = {f"{c}_L{lid}": weight_vectors[(c, lid)] for c, lid in weight_vectors}
np.savez_compressed(str(wv_path), **wv_dict)
print(f"Saved: {wv_path} ({len(wv_dict)} vectors)")

hf.close()
print("\nV2 complete.")